In [2]:
import pypsa
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import geopandas as gpd
from pypsa.plot import add_legend_lines, add_legend_patches, add_legend_semicircles
import yaml
from pathlib import Path
import pandas as pd
from scripts._helpers import (
    configure_logging,
    get_snapshots,
    load_cutout,
    set_scenario_config,
)

**Set Up**

In [3]:
fn = 'resources/DK_test/networks/base_s_2__12h_2050.nc'



In [4]:
n= pypsa.Network(fn)

config = yaml.safe_load(Path("config/config.denmark.yaml").read_text())
industrial_production = pd.read_csv("resources/DK_test/industrial_production_base_s_2_2050.csv", index_col=0)
industry_sector_ratios = pd.read_csv("resources/DK_test/industry_sector_ratios.csv", index_col=0)



INFO:pypsa.network.io:New version 1.0.7 available! (Current: 0.35.2)
INFO:pypsa.network.io:Imported network 'Unnamed Network' has buses, carriers, generators, global_constraints, links, loads, stores


In [5]:
p = Path(fn)  
try:
    if p.exists():
        p.unlink()
        print(f"Deleted {p}")
    else:
        print(f"File not found: {p}")
except Exception as e:
    print(f"Failed to delete {p}: {e}")

Deleted resources/DK_test/networks/base_s_2__12h_2050.nc


**Options**

In [ ]:
ongrid=False
cluster_cost_reduction=0.3
cluster_size=800                                        #MW, it's the maximum installable capacity for each renewable in renewables in each country cluster
renewables={"solar-hsat","solar","onwind"}
co2_storage_treshold=200*1e3                            #200 ktonnes CO2 stored from industry threshold for making the node eligible for carbon clusters

**CO2 Use Availability from Industries and Node Definition**

In [7]:
def check_industrial_production_columns(industrial_production, industry_sector_ratios):
    """Check that the columns in industrial_production and industry_sector_ratios match."""

    extra_columns = set(industry_sector_ratios.columns) - set(industrial_production.columns)

    if extra_columns:
        raise ValueError(
            f"{', '.join(extra_columns)} found in industry_sector_ratios but not in industrial_production"
        )

    extra_columns = set(industrial_production.columns) - set(industry_sector_ratios.columns)  

    if extra_columns:
        raise ValueError(
            f"{', '.join(extra_columns)} found in industrial_production but not in industry_sector_ratios"
        )


    industry_sector_ratios = industry_sector_ratios[industrial_production.columns]
    return industrial_production, industry_sector_ratios

industrial_production, industry_sector_ratios = check_industrial_production_columns(industrial_production, industry_sector_ratios)

In [8]:
def find_nodes_eligible_for_carbon_clusters(n, industrial_production, industry_sector_ratios, co2_storage_treshold):    

    biomass_by_industry_by_node=pd.DataFrame()
    methane_by_industry_by_node=pd.DataFrame()
    process_emissions_by_industry_by_node=pd.DataFrame()
    carbon_available_by_industry_by_node=pd.DataFrame()
    nodes_with_carbon_clusters=[]


    for node in industrial_production.index:
        eff_biomass_CC = n.links.loc[
                n.links.index.str.contains(f'{node} solid biomass for industry CC'),
                'efficiency3'
            ].iloc[0]*n.links.loc[
                n.links.index.str.contains(f'{node} solid biomass for industry CC'),
                'efficiency'
            ].iloc[0]

        eff_methane_CC = n.links.loc[
                n.links.index.str.contains(f'{node} gas for industry CC'),
                'efficiency3'
            ].iloc[0]*n.links.loc[
                n.links.index.str.contains(f'{node} gas for industry CC'),
                'efficiency'
            ].iloc[0]

            
        for sector in industrial_production.columns:
            biomass_by_industry_by_node.loc[node,sector]=industrial_production.loc[node,sector]*industry_sector_ratios.loc['biomass',sector]*1e3 # MWh = ktonnes/year *MWh/tonne *tonnes/ktonnes
            methane_by_industry_by_node.loc[node,sector]=industrial_production.loc[node,sector]*industry_sector_ratios.loc['methane',sector]*1e3 # tonnesCO2 = ktonnes/year *MWh/tonne *tonnes/ktonnes
            process_emissions_by_industry_by_node.loc[node,sector]= industrial_production.loc[node,sector]*industry_sector_ratios.loc['process emission',sector]*1e3+industrial_production.loc[node,sector]*industry_sector_ratios.loc['process emission from feedstock',sector]*1e3 # tonnesCO2 = ktonnes/year * tonnesCO2/tonne * tonnes/ktonnes

            

            carbon_from_biomass=biomass_by_industry_by_node.loc[node,sector]*eff_biomass_CC
            carbon_from_methane=methane_by_industry_by_node.loc[node,sector]*eff_methane_CC


        

            carbon_available_by_industry_by_node.loc[node,sector]=carbon_from_biomass+carbon_from_methane+process_emissions_by_industry_by_node.loc[node,sector]

            if (carbon_available_by_industry_by_node.loc[node].sum()>co2_storage_treshold and node not in nodes_with_carbon_clusters):
                nodes_with_carbon_clusters.append(node)

    return nodes_with_carbon_clusters, carbon_available_by_industry_by_node, process_emissions_by_industry_by_node


nodes_with_carbon_clusters,  carbon_available_by_industry_by_node, process_emissions_by_industry_by_node  =find_nodes_eligible_for_carbon_clusters(n, industrial_production, industry_sector_ratios, co2_storage_treshold)

nodes_with_carbon_clusters



['DK0 0', 'DK1 0']

In [9]:
process_emissions_by_industry_by_node.sum(axis=1)

DK0 0    1.511454e+06
DK1 0    2.180916e+04
dtype: float64

In [10]:
carbon_available_by_industry_by_node.sum(axis=1)

DK0 0    3.226754e+06
DK1 0    6.954475e+05
dtype: float64

In [11]:
n.loads.loc[n.loads.index.str.contains("process emissions", na=False)]

,bus,carrier,type,p_set,q_set,sign,active
Load,,,,,,,
DK0 0 process emissions,DK0 0 process emissions,process emissions,,-172.374429,0.0,-1.0,True
DK1 0 process emissions,DK1 0 process emissions,process emissions,,-2.283105,0.0,-1.0,True


In [14]:
def assign_co2_bus_to_carbon_clusters(n, nodes_with_carbon_clusters):

    for node in nodes_with_carbon_clusters:

        #define the CO2 bus for the carbon cluster
        bus = f"{node} co2 stored"

        n.add(
            "Bus",
            name=bus + " cluster",
            v_nom=n.buses.at[bus, "v_nom"],
            x=n.buses.at[bus, "x"],
            y=n.buses.at[bus, "y"],
            unit=n.buses.at[bus, "unit"],
            location=n.buses.at[bus, "location"],
            country=n.buses.at[bus, "country"],
            carrier=n.buses.at[bus, "carrier"],
            control=n.buses.at[bus, "control"],
            substation_lv=n.buses.at[bus, "substation_lv"],
            substation_off=n.buses.at[bus, "substation_off"],
            overwrite=True,
        )

        #connect the industry processes CC to the carbon cluster co2 bus

        link_name = f"{node} process emissions CC"

        n.add(
            "Link",
            name=link_name,
            bus0=n.links.at[link_name, "bus0"],
            bus1=n.links.at[link_name, "bus1"],
            bus2=bus + " cluster",
            bus3=n.links.at[link_name, "bus3"],
            bus4=n.links.at[link_name, "bus4"],
            p_nom_extendable=n.links.at[link_name, "p_nom_extendable"],
            p_min_pu=n.links.at[link_name, "p_min_pu"],
            carrier=n.links.at[link_name, "carrier"],
            efficiency=n.links.at[link_name, "efficiency"],
            efficiency2=n.links.at[link_name, "efficiency2"],
            efficiency3=n.links.at[link_name, "efficiency3"],
            efficiency4=n.links.at[link_name, "efficiency4"],
            capital_cost=n.links.at[link_name, "capital_cost"],
            marginal_cost=n.links.at[link_name, "marginal_cost"],
            lifetime=n.links.at[link_name, "lifetime"],
            reversed=False,
            overwrite=True,
        )


        link_name = f"{node} gas for industry CC"

        n.add(
            "Link",
            name=link_name,
            bus0=n.links.at[link_name, "bus0"],
            bus1=n.links.at[link_name, "bus1"],
            bus2=n.links.at[link_name, "bus2"],
            bus3=bus + " cluster",
            bus4=n.links.at[link_name, "bus4"],
            p_nom_extendable=n.links.at[link_name, "p_nom_extendable"],
            p_min_pu=n.links.at[link_name, "p_min_pu"],
            carrier=n.links.at[link_name, "carrier"],
            efficiency=n.links.at[link_name, "efficiency"],
            efficiency2=n.links.at[link_name, "efficiency2"],
            efficiency3=n.links.at[link_name, "efficiency3"],
            efficiency4=n.links.at[link_name, "efficiency4"],
            capital_cost=n.links.at[link_name, "capital_cost"],
            marginal_cost=n.links.at[link_name, "marginal_cost"],
            lifetime=n.links.at[link_name, "lifetime"],
            reversed=False,
            overwrite=True,
        )

        link_name = f"{node} solid biomass for industry CC"

        n.add(
            "Link",
            name=link_name,
            bus0=n.links.at[link_name, "bus0"],
            bus1=n.links.at[link_name, "bus1"],
            bus2=n.links.at[link_name, "bus2"],
            bus3=bus + " cluster",
            bus4=n.links.at[link_name, "bus4"],
            p_nom_extendable=n.links.at[link_name, "p_nom_extendable"],
            p_min_pu=n.links.at[link_name, "p_min_pu"],
            carrier=n.links.at[link_name, "carrier"],
            efficiency=n.links.at[link_name, "efficiency"],
            efficiency2=n.links.at[link_name, "efficiency2"],
            efficiency3=n.links.at[link_name, "efficiency3"],
            efficiency4=n.links.at[link_name, "efficiency4"],
            capital_cost=n.links.at[link_name, "capital_cost"],
            marginal_cost=n.links.at[link_name, "marginal_cost"],
            lifetime=n.links.at[link_name, "lifetime"],
            reversed=False,
            overwrite=True,
        )

    return n

n = assign_co2_bus_to_carbon_clusters(n, nodes_with_carbon_clusters)



**Buses and Generators of the Cluster Addition**

In [15]:
def assign_cluster_generators_and_electricity_buses_to_carbon_clusters(n, config, cluster_size, cluster_cost_reduction, renewables, nodes_with_clusters):
    
    nodes_renewables_cf = {}                #dictionary of dataframes by node and renewable type, sorting the generators by average capacity factor (ascending order)
    clusters_generators={}                      #dictionary of dataframes by node and renewable type, containing the generators assigned to the cluster  

    for node in nodes_with_clusters:
        for renewable in renewables:

            nodes_renewables_cf[(node, renewable)] = pd.DataFrame(
                index=n.generators['p_nom_max'].loc[n.generators.index.astype(str).str.contains(rf"{node}.*{renewable}$")].index,
                columns=["p_max_pu","p_nom_max"]  
            )

            clusters_generators[(node, renewable)] = pd.DataFrame()

            #we are considering the highest mean p_min_pu to determine the best generators per renewable available

            nodes_renewables_cf[(node, renewable)] ["p_max_pu"] = n.generators_t['p_max_pu'].loc[:, n.generators_t['p_max_pu'].columns.astype(str).str.contains(rf"{node}.*{renewable}$")].mean()
            nodes_renewables_cf[(node, renewable)] ["p_nom_max"] = n.generators['p_nom_max'].loc[n.generators.index.astype(str).str.contains(rf"{node}.*{renewable}$")]

            nodes_renewables_cf[(node, renewable)] = nodes_renewables_cf[(node, renewable)].sort_values("p_max_pu", ascending=True)


            #print(nodes_renewables_cf[(country, renewable)])

            number_gen=0
            
            insufficient_generators = False

            while nodes_renewables_cf[(node, renewable)].iloc[0:number_gen+1]["p_nom_max"].sum() <= cluster_size:

                if number_gen >= len(nodes_renewables_cf[(node, renewable)]):
                    print(f"Not enough {renewable} generators at node {node} to reach cluster_size.")
                    insufficient_generators = True
                    break

                number_gen += 1

            if insufficient_generators:
                continue

            #print(f"{renewable} generators in cluster: {number_gen+1}")

            clusters_generators[(node, renewable)]  = n.generators.loc[nodes_renewables_cf[(node, renewable)].index[0:number_gen+1]]
            remaining_capacity = nodes_renewables_cf[(node, renewable)].iloc[0:number_gen+1]["p_nom_max"].sum() - cluster_size
            #nodes_renewables_cf[(country, renewable)].iloc[number_gen]["p_nom_max"] = remaining_capacity maybe it is better to do this step later

            print(f"Remaining top {renewable} capacity outside the cluster: {remaining_capacity} MW")

            
            clusters_generators[(node, renewable)].loc[clusters_generators[(node, renewable)].index[number_gen], "p_nom_max"] = cluster_size - clusters_generators[(node, renewable)].loc[clusters_generators[(node, renewable)].index[0:number_gen],"p_nom_max"].sum()

            print(f"Capacity of the last {renewable} generator adjusted to fit cluster size: {clusters_generators[(node, renewable)].loc[clusters_generators[(node, renewable)].index[number_gen], 'p_nom_max']} MW")

            print(clusters_generators[(node, renewable)])

            for idx in clusters_generators[(node, renewable)].index:

                ### Electricity bus and generators ###

                if not n.buses.index.str.contains(rf"{clusters_generators[(node, renewable)].loc[idx].bus + " cluster"}$").any():
        
                    n.add(
                        "Bus",
                        name=clusters_generators[(node, renewable)].loc[idx].bus + " cluster",
                        v_nom=n.buses.at[clusters_generators[(node, renewable)].loc[idx].bus, "v_nom"],
                        x=n.buses.at[clusters_generators[(node, renewable)].loc[idx].bus, "x"],
                        y=n.buses.at[clusters_generators[(node, renewable)].loc[idx].bus, "y"],
                        unit=n.buses.at[clusters_generators[(node, renewable)].loc[idx].bus, "unit"],
                        location=n.buses.at[clusters_generators[(node, renewable)].loc[idx].bus, "location"],
                        country=n.buses.at[clusters_generators[(node, renewable)].loc[idx].bus, "country"],
                        carrier=n.buses.at[clusters_generators[(node, renewable)].loc[idx].bus, "carrier"],
                        control=n.buses.at[clusters_generators[(node, renewable)].loc[idx].bus, "control"],
                        substation_lv=n.buses.at[clusters_generators[(node, renewable)].loc[idx].bus, "substation_lv"],
                        substation_off=n.buses.at[clusters_generators[(node, renewable)].loc[idx].bus, "substation_off"],
                    )

                n.add(
                    "Generator",
                    name=clusters_generators[(node, renewable)].loc[idx].name + " cluster",
                    bus=clusters_generators[(node, renewable)].loc[idx].bus + " cluster",
                    carrier=clusters_generators[(node, renewable)].loc[idx].carrier,
                    p_nom_max=clusters_generators[(node, renewable)].loc[idx].p_nom_max,
                    p_max_pu=clusters_generators[(node, renewable)].loc[idx].p_max_pu,
                    marginal_cost=clusters_generators[(node, renewable)].loc[idx].marginal_cost*(1-cluster_cost_reduction),
                    capital_cost=clusters_generators[(node, renewable)].loc[idx].capital_cost*(1-cluster_cost_reduction),
                    efficiency=clusters_generators[(node, renewable)].loc[idx].efficiency,
                    location=clusters_generators[(node, renewable)].loc[idx].location,
                    unit=clusters_generators[(node, renewable)].loc[idx].unit,
                    p_nom_extendable=True,
                    p_nom_min=100,
                    overwrite=True,)


                
                
                n.generators_t['p_max_pu'][clusters_generators[(node, renewable)].loc[idx].name + " cluster"] = n.generators_t['p_max_pu'][clusters_generators[(node, renewable)].loc[idx].name]


                ### H2 bus ##

                if not n.buses.index.str.contains(rf"{clusters_generators[(node, renewable)].loc[idx].bus + " H2 cluster"}$").any():

                    n.add(
                        "Bus",
                        name=clusters_generators[(node, renewable)].loc[idx].bus + " H2 cluster",
                        v_nom=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + " H2"}", "v_nom"],
                        x=n.buses.at[rf"{clusters_generators[(node,renewable)].loc[idx].bus + " H2"}", "x"],
                        y=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + " H2"}", "y"],
                        unit=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + " H2"}", "unit"],
                        location=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + " H2"}", "location"],
                        country=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + " H2"}", "country"],
                        carrier=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + " H2"}", "carrier"],
                        control=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + " H2"}", "control"],
                        substation_lv=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + " H2"}", "substation_lv"],
                        substation_off=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + " H2"}", "substation_off"],
                    )
                
                ### Batteries bus ###

                if not n.buses.index.str.contains(rf"{clusters_generators[(node, renewable)].loc[idx].bus + " battery cluster"}$").any():

                    n.add(
                        "Bus",
                        name=clusters_generators[(node, renewable)].loc[idx].bus + " battery cluster",
                        v_nom=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + " battery"}", "v_nom"],
                        x=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + " battery"}", "x"],
                        y=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + " battery"}", "y"],
                        unit=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + " battery"}", "unit"],
                        location=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + " battery"}", "location"],
                        country=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + " battery"}", "country"],
                        carrier=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + " battery"}", "carrier"],
                        control=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + " battery"}", "control"],
                        substation_lv=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + " battery"}", "substation_lv"],
                        substation_off=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + " battery"}", "substation_off"],
                    )



                if idx == nodes_renewables_cf[(node, renewable)].iloc[number_gen].name:
                    n.generators.loc[n.generators.index == idx, "p_nom_max"] = nodes_renewables_cf[(node, renewable)].iloc[0:number_gen+1]["p_nom_max"].sum() - cluster_size

                    print(f"Residual capacity of generator {clusters_generators[(node, renewable)].loc[idx].name} is {n.generators.loc[n.generators.index == idx, 'p_nom_max']} MW")
                
                else:


                    n.remove(
                            "Generator",
                            name=clusters_generators[(node, renewable)].loc[idx].name,
                    )



            

    return n

n = assign_cluster_generators_and_electricity_buses_to_carbon_clusters(n, config, cluster_size, cluster_cost_reduction, renewables, nodes_with_carbon_clusters)
            



            

            





        




Remaining top solar capacity outside the cluster: 31289.17326629335 MW
Capacity of the last solar generator adjusted to fit cluster size: 800.0 MW
                 bus control type  p_nom  p_nom_mod  p_nom_extendable  \
Generator                                                               
DK0 0 0 solar  DK0 0      PQ       800.2        0.0              True   

               p_nom_min  p_nom_max  p_min_pu  p_max_pu  ...  up_time_before  \
Generator                                                ...                   
DK0 0 0 solar      800.2      800.0       0.0       1.0  ...               1   

               down_time_before  ramp_limit_up  ramp_limit_down  \
Generator                                                         
DK0 0 0 solar                 0            NaN              NaN   

               ramp_limit_start_up ramp_limit_shut_down  weight  p_nom_opt  \
Generator                                                                    
DK0 0 0 solar                  1.0

In [16]:
n.buses

,v_nom,type,x,y,carrier,unit,location,v_mag_pu_set,v_mag_pu_min,v_mag_pu_max,control,generator,sub_network,country,substation_lv,substation_off
Bus,,,,,,,,,,,,,,,,
DK0 0,380.0,,9.648393,55.893047,AC,MWh_el,DK0 0,1.0,0.0,inf,Slack,,,DK,1.0,1.0
DK1 0,380.0,,12.303316,55.515974,AC,MWh_el,DK1 0,1.0,0.0,inf,Slack,,,DK,1.0,1.0
EU,1.0,,-5.500000,46.000000,none,,EU,1.0,0.0,inf,PQ,,,,NaN,NaN
co2 atmosphere,1.0,,-5.500000,46.000000,co2,t_co2,EU,1.0,0.0,inf,PQ,,,,NaN,NaN
DK0 0 co2 stored,1.0,,9.648393,55.893047,co2 stored,t_co2,DK0 0,1.0,0.0,inf,PQ,,,DK,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
DK0 0 H2 cluster,1.0,,9.648393,55.893047,H2,MWh_LHV,DK0 0,1.0,0.0,inf,PQ,,,DK,NaN,NaN
DK0 0 battery cluster,1.0,,9.648393,55.893047,battery,MWh_el,DK0 0,1.0,0.0,inf,PQ,,,DK,NaN,NaN
DK1 0 cluster,380.0,,12.303316,55.515974,AC,MWh_el,DK1 0,1.0,0.0,inf,Slack,,,DK,1.0,1.0


**Links of the Cluster Addition**

In [17]:
n.links

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,underground,under_construction,tags,geometry,dc,underwater_fraction,energy to power ratio,location,reversed,length_original
Link,,,,,,,,,,,,,,,,,,,,,
relation/5487095-400-DC,DK0 0,DK1 0,,DC,0.976096,True,0,inf,600.0,0.0,...,1.0,0.0,relation/5487095,LINESTRING (10.505724427906852 55.365970143543...,1.0,0.560249,NaN,,False,171.53249
DK0 0 co2 sequestered,DK0 0 co2 stored,DK0 0 co2 sequestered,,co2 sequestered,1.000000,True,0,inf,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.00000
DK1 0 co2 sequestered,DK1 0 co2 stored,DK1 0 co2 sequestered,,co2 sequestered,1.000000,True,0,inf,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.00000
DK0 0 OCGT,DK0 0 gas,DK0 0,,OCGT,0.430000,True,0,25.0,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.00000
DK1 0 OCGT,DK1 0 gas,DK1 0,,OCGT,0.430000,True,0,25.0,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.00000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
DK0 0 gas for industry CC,DK0 0 gas,DK0 0 gas for industry,,gas for industry CC,0.900000,True,0,25.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
DK0 0 solid biomass for industry CC,DK0 0 solid biomass,DK0 0 solid biomass for industry,,solid biomass for industry CC,0.900000,True,0,25.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
DK1 0 process emissions CC,DK1 0 process emissions,co2 atmosphere,,process emissions CC,0.050000,True,0,25.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN


In [18]:
def add_cluster_links(n, nodes_with_clusters, cluster_cost_reduction, ongrid):

    for node in nodes_with_clusters:

        ### H2 Electrolysis ###

        link_name = f"{node} H2 Electrolysis"

        n.add(
            "Link",
            name=link_name + " cluster",
            bus0=n.links.at[link_name, "bus0"] + " cluster",
            bus1=n.links.at[link_name, "bus1"] + " cluster",
            p_nom_extendable=n.links.at[link_name, "p_nom_extendable"],
            carrier=n.links.at[link_name, "carrier"],
            efficiency=n.links.at[link_name, "efficiency"],
            capital_cost=n.links.at[link_name, "capital_cost"]*(1-cluster_cost_reduction),
            marginal_cost=n.links.at[link_name, "marginal_cost"]*(1-cluster_cost_reduction),
            lifetime=n.links.at[link_name, "lifetime"],
            reversed=False,
            overwrite=True,
        )

        ### Methanolization ###

        link_name = f"{node} methanolisation"

        n.add(
            "Link",
            name=link_name + " cluster",
            bus0=n.links.at[link_name, "bus0"] + " cluster",
            bus1=n.links.at[link_name, "bus1"],
            bus2=n.links.at[link_name, "bus2"] + " cluster",
            bus3=n.links.at[link_name, "bus3"] + " cluster",
            bus4=n.links.at[link_name, "bus4"],
            p_nom_extendable=n.links.at[link_name, "p_nom_extendable"],
            p_min_pu=n.links.at[link_name, "p_min_pu"],
            carrier=n.links.at[link_name, "carrier"],
            efficiency=n.links.at[link_name, "efficiency"],
            efficiency2=n.links.at[link_name, "efficiency2"],
            efficiency3=n.links.at[link_name, "efficiency3"],
            efficiency4=n.links.at[link_name, "efficiency4"],
            capital_cost=n.links.at[link_name, "capital_cost"]*(1-cluster_cost_reduction),
            marginal_cost=n.links.at[link_name, "marginal_cost"]*(1-cluster_cost_reduction),
            lifetime=n.links.at[link_name, "lifetime"],
            reversed=False,
            overwrite=True,
        )


    if ongrid==True :

        ### Electricity connection to grid ###

        link_name = f"{node} electricity cluster"
        
        n.add(
            "Link",
            name=link_name,
            bus0=f"{node} cluster",
            bus1=f"{node}",
            carrier=n.buses.at[f"{node}", "carrier"],  
            p_nom_extendable=True,
            efficiency=1.0,
            capital_cost=0.0,
            marginal_cost=0.0,
            reversed=False,
            overwrite=True,
        )

        link_name = f"{node} electricity cluster back"
        n.add(
            "Link",
            name=link_name,
            bus0=f"{node}",
            bus1=f"{node} cluster",
            carrier=n.buses.at[f"{node}", "carrier"],  
            p_nom_extendable=True,
            efficiency=1.0,
            capital_cost=0.0,
            marginal_cost=0.0,
            reversed=True,
            overwrite=True,
        )

    else:
        if f"{node} cluster electricity" in n.links.index:
            n.remove(
                "Link",
                name=f"{node} cluster electricity",
            )
        if f"{node} cluster electricity back" in n.links.index:
            n.remove(
                "Link",
                name=f"{node} cluster electricity back",
            )

    return n

n = add_cluster_links(n, nodes_with_carbon_clusters, cluster_cost_reduction, ongrid)



        


        

In [19]:
n.links.loc[n.links.index.str.contains("methanolisation")]

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,underground,under_construction,tags,geometry,dc,underwater_fraction,energy to power ratio,location,reversed,length_original
Link,,,,,,,,,,,,,,,,,,,,,
DK0 0 methanolisation,DK0 0 H2,EU methanol,,methanolisation,0.8787,True,0,20.0,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.0
DK1 0 methanolisation,DK1 0 H2,EU methanol,,methanolisation,0.8787,True,0,20.0,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.0
DK0 0 methanolisation cluster,DK0 0 H2 cluster,EU methanol,,methanolisation,0.8787,True,0,20.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
DK1 0 methanolisation cluster,DK1 0 H2 cluster,EU methanol,,methanolisation,0.8787,True,0,20.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN


**Storages of the Cluster Addition**

In [20]:
def add_cluster_storages(n, nodes_with_clusters, cluster_cost_reduction):

    for node in nodes_with_clusters:

        link_name = f"{node} H2 Store"

    
        n.add("Store",
            name=link_name + " cluster",
            bus=n.stores.at[link_name, "bus"] + " cluster",
            carrier=n.stores.at[link_name, "carrier"],
            e_nom_extendable=True,
            capital_cost=n.stores.at[link_name, "capital_cost"]*(1-cluster_cost_reduction),
            marginal_cost=n.stores.at[link_name, "marginal_cost"]*(1-cluster_cost_reduction),
            e_initial_per_period=n.stores.at[link_name, "e_initial_per_period"],
            e_cyclic=n.stores.at[link_name, "e_cyclic"],
            e_cyclic_per_period=n.stores.at[link_name, "e_cyclic_per_period"],
            overwrite=True,
            )
        
        link_name = f"{node} battery"


        n.add(
                "Link",
                name=link_name + " charger cluster",
                bus0=f"{node} cluster",
                bus1=f"{node} battery cluster",
                carrier=n.buses.at[link_name, "carrier"],   
                p_nom_extendable=True,
                efficiency=1.0,
                capital_cost=0.0,
                marginal_cost=0.0,
                reversed=False,
                overwrite=True,
            )
        n.add(
                "Link",
                name=link_name + " discharger cluster",
                bus0=f"{node} battery cluster",
                bus1=f"{node} cluster",
                carrier=n.buses.at[link_name, "carrier"],
                p_nom_extendable=True,
                efficiency=1.0,
                capital_cost=0.0,
                marginal_cost=0.0,
                reversed=True,
                overwrite=True,
            )

        n.add("Store",
            name=link_name + " cluster" ,
            bus=f"{node} battery cluster",
            carrier=n.stores.at[link_name, "carrier"],
            e_nom_extendable=True,
            capital_cost=n.stores.at[link_name, "capital_cost"]*(1-cluster_cost_reduction),
            marginal_cost=n.stores.at[link_name, "marginal_cost"]*(1-cluster_cost_reduction),
            e_initial_per_period=n.stores.at[link_name, "e_initial_per_period"],
            e_cyclic=n.stores.at[link_name, "e_cyclic"],
            e_cyclic_per_period=n.stores.at[link_name, "e_cyclic_per_period"],
            overwrite=True,
            )
    return n

n = add_cluster_storages(n, nodes_with_carbon_clusters, cluster_cost_reduction)





In [21]:
n.links["reversed"] = n.links["reversed"].fillna(False).astype(bool)


**Printing to Check**

In [22]:
n.links.loc[n.links["bus1"]=='EU methanol']

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,underground,under_construction,tags,geometry,dc,underwater_fraction,energy to power ratio,location,reversed,length_original
Link,,,,,,,,,,,,,,,,,,,,,
DK0 0 solid biomass biomass-to-methanol,DK0 0 solid biomass,EU methanol,,biomass-to-methanol,0.6500,True,0,20.0,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.0
DK1 0 solid biomass biomass-to-methanol,DK1 0 solid biomass,EU methanol,,biomass-to-methanol,0.6500,True,0,20.0,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.0
DK0 0 methanolisation,DK0 0 H2,EU methanol,,methanolisation,0.8787,True,0,20.0,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.0
DK1 0 methanolisation,DK1 0 H2,EU methanol,,methanolisation,0.8787,True,0,20.0,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.0
DK0 0 methanolisation cluster,DK0 0 H2 cluster,EU methanol,,methanolisation,0.8787,True,0,20.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
DK1 0 methanolisation cluster,DK1 0 H2 cluster,EU methanol,,methanolisation,0.8787,True,0,20.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN


In [23]:
n.links.loc[n.links.index.str.contains("Electrolysis")]

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,underground,under_construction,tags,geometry,dc,underwater_fraction,energy to power ratio,location,reversed,length_original
Link,,,,,,,,,,,,,,,,,,,,,
DK0 0 H2 Electrolysis,DK0 0,DK0 0 H2,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.0
DK1 0 H2 Electrolysis,DK1 0,DK1 0 H2,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.0
DK0 0 H2 Electrolysis cluster,DK0 0 cluster,DK0 0 H2 cluster,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
DK1 0 H2 Electrolysis cluster,DK1 0 cluster,DK1 0 H2 cluster,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN


In [24]:
n.links.loc[n.links["carrier"].str.contains('H2 Electrolysis')]

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,underground,under_construction,tags,geometry,dc,underwater_fraction,energy to power ratio,location,reversed,length_original
Link,,,,,,,,,,,,,,,,,,,,,
DK0 0 H2 Electrolysis,DK0 0,DK0 0 H2,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.0
DK1 0 H2 Electrolysis,DK1 0,DK1 0 H2,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.0
DK0 0 H2 Electrolysis cluster,DK0 0 cluster,DK0 0 H2 cluster,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
DK1 0 H2 Electrolysis cluster,DK1 0 cluster,DK1 0 H2 cluster,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN


In [25]:
n.buses.loc[n.buses.index.str.contains("cluster")]

,v_nom,type,x,y,carrier,unit,location,v_mag_pu_set,v_mag_pu_min,v_mag_pu_max,control,generator,sub_network,country,substation_lv,substation_off
Bus,,,,,,,,,,,,,,,,
DK0 0 co2 stored cluster,1.0,,9.648393,55.893047,co2 stored,t_co2,DK0 0,1.0,0.0,inf,PQ,,,DK,NaN,NaN
DK1 0 co2 stored cluster,1.0,,12.303316,55.515974,co2 stored,t_co2,DK1 0,1.0,0.0,inf,PQ,,,DK,NaN,NaN
DK0 0 cluster,380.0,,9.648393,55.893047,AC,MWh_el,DK0 0,1.0,0.0,inf,Slack,,,DK,1.0,1.0
DK0 0 H2 cluster,1.0,,9.648393,55.893047,H2,MWh_LHV,DK0 0,1.0,0.0,inf,PQ,,,DK,NaN,NaN
DK0 0 battery cluster,1.0,,9.648393,55.893047,battery,MWh_el,DK0 0,1.0,0.0,inf,PQ,,,DK,NaN,NaN
DK1 0 cluster,380.0,,12.303316,55.515974,AC,MWh_el,DK1 0,1.0,0.0,inf,Slack,,,DK,1.0,1.0
DK1 0 H2 cluster,1.0,,12.303316,55.515974,H2,MWh_LHV,DK1 0,1.0,0.0,inf,PQ,,,DK,NaN,NaN
DK1 0 battery cluster,1.0,,12.303316,55.515974,battery,MWh_el,DK1 0,1.0,0.0,inf,PQ,,,DK,NaN,NaN


In [26]:
n.stores.loc[n.stores.index.str.contains("cluster")]



,bus,type,carrier,e_nom,e_nom_mod,e_nom_extendable,e_nom_min,e_nom_max,e_min_pu,e_max_pu,...,marginal_cost,marginal_cost_quadratic,marginal_cost_storage,capital_cost,standing_loss,active,build_year,lifetime,e_nom_opt,location
Store,,,,,,,,,,,,,,,,,,,,,
DK0 0 H2 Store cluster,DK0 0 H2 cluster,,H2 Store,0.0,0.0,True,0.0,inf,0.0,1.0,...,0.0,0.0,0.0,62.601045,0.0,True,0,inf,0.0,NaN
DK0 0 battery cluster,DK0 0 battery cluster,,battery,0.0,0.0,True,0.0,inf,0.0,1.0,...,0.0,0.0,0.0,4499.018028,0.0,True,0,inf,0.0,NaN
DK1 0 H2 Store cluster,DK1 0 H2 cluster,,H2 Store,0.0,0.0,True,0.0,inf,0.0,1.0,...,0.0,0.0,0.0,62.601045,0.0,True,0,inf,0.0,NaN
DK1 0 battery cluster,DK1 0 battery cluster,,battery,0.0,0.0,True,0.0,inf,0.0,1.0,...,0.0,0.0,0.0,4499.018028,0.0,True,0,inf,0.0,NaN


In [27]:
n.links.loc[n.links["carrier"]=='DC']

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,underground,under_construction,tags,geometry,dc,underwater_fraction,energy to power ratio,location,reversed,length_original
Link,,,,,,,,,,,,,,,,,,,,,
relation/5487095-400-DC,DK0 0,DK1 0,,DC,0.976096,True,0,inf,600.0,0.0,...,1.0,0.0,relation/5487095,LINESTRING (10.505724427906852 55.365970143543...,1.0,0.560249,NaN,,False,171.53249
relation/5487095-400-DC-reversed,DK1 0,DK0 0,,DC,0.976096,True,0,inf,600.0,0.0,...,1.0,0.0,relation/5487095,LINESTRING (10.505724427906852 55.365970143543...,1.0,0.560249,NaN,,True,171.53249


In [28]:
n.stores

,bus,type,carrier,e_nom,e_nom_mod,e_nom_extendable,e_nom_min,e_nom_max,e_min_pu,e_max_pu,...,marginal_cost,marginal_cost_quadratic,marginal_cost_storage,capital_cost,standing_loss,active,build_year,lifetime,e_nom_opt,location
Store,,,,,,,,,,,,,,,,,,,,,
co2 atmosphere,co2 atmosphere,,co2,0.000000,0.0,True,0.0,inf,-1.0,1.0,...,0.0,0.0,0.0,0.000000,0.000000,True,0,inf,0.0,
DK0 0 co2 stored,DK0 0 co2 stored,,co2 stored,0.000000,0.0,True,0.0,inf,0.0,1.0,...,0.0,0.0,0.0,247.607546,0.000000,True,0,inf,0.0,
DK1 0 co2 stored,DK1 0 co2 stored,,co2 stored,0.000000,0.0,True,0.0,inf,0.0,1.0,...,0.0,0.0,0.0,247.607546,0.000000,True,0,inf,0.0,
DK0 0 co2 sequestered,DK0 0 co2 sequestered,,co2 sequestered,0.000000,0.0,True,0.0,7.374268e+08,0.0,1.0,...,-0.1,0.0,0.0,30.000000,0.000000,True,0,50.0,0.0,
DK1 0 co2 sequestered,DK1 0 co2 sequestered,,co2 sequestered,0.000000,0.0,True,0.0,8.677499e+07,0.0,1.0,...,-0.1,0.0,0.0,30.000000,0.000000,True,0,50.0,0.0,
DK0 0 gas Store,DK0 0 gas,,gas,0.000000,0.0,True,3805600.0,inf,0.0,1.0,...,0.0,0.0,0.0,17.851178,0.000000,True,0,inf,0.0,
DK1 0 gas Store,DK1 0 gas,,gas,0.000000,0.0,True,6334336.0,inf,0.0,1.0,...,0.0,0.0,0.0,17.851178,0.000000,True,0,inf,0.0,
DK0 0 H2 Store,DK0 0 H2,,H2 Store,0.000000,0.0,True,0.0,2.013398e+08,0.0,1.0,...,0.0,0.0,0.0,89.430064,0.000000,True,0,100.0,0.0,
DK1 0 H2 Store,DK1 0 H2,,H2 Store,0.000000,0.0,True,0.0,2.879317e+06,0.0,1.0,...,0.0,0.0,0.0,89.430064,0.000000,True,0,100.0,0.0,


In [29]:
n.links.loc[n.links["bus0"]=='EU methanol']

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,underground,under_construction,tags,geometry,dc,underwater_fraction,energy to power ratio,location,reversed,length_original
Link,,,,,,,,,,,,,,,,,,,,,
DK0 0 OCGT methanol,EU methanol,DK0 0,,OCGT methanol,0.43,True,0,25.0,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.0
DK1 0 OCGT methanol,EU methanol,DK1 0,,OCGT methanol,0.43,True,0,25.0,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.0
EU industry methanol,EU methanol,EU industry methanol,,industry methanol,1.00,True,0,inf,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.0
EU shipping methanol,EU methanol,EU shipping methanol,,shipping methanol,1.00,True,0,inf,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.0


In [30]:
n.carriers

,co2_emissions,color,nice_name,max_growth,max_relative_growth
Carrier,,,,,
AC,0.0,#70af1d,AC,inf,0.0
DC,0.0,#8a1caf,DC,inf,0.0
onwind,0.0,#235ebc,Onshore Wind,inf,0.0
offwind-float,0.0,#b5e2fa,Offshore Wind (Floating),inf,0.0
offwind-dc,0.0,#74c6f2,Offshore Wind (DC),inf,0.0
...,...,...,...,...,...
home battery charger,0.0,#5e8032,home battery charger,inf,0.0
urban central heat vent,0.0,#a74747,urban central heat vent,inf,0.0
solar rooftop,0.0,#ffea80,solar rooftop,inf,0.0


In [31]:
n.global_constraints

,type,investment_period,carrier_attribute,sense,constant,mu
GlobalConstraint,,,,,,
lv_limit,transmission_volume_expansion_limit,NaN,"AC, DC",<=,1.029195e+05,0.0
biomass limit,operational_limit,NaN,solid biomass,<=,1.219759e+07,0.0
CO2Limit,co2_atmosphere,NaN,co2_emissions,<=,0.000000e+00,0.0


In [32]:
n.loads

,bus,carrier,type,p_set,q_set,sign,active
Load,,,,,,,
DK0 0,DK0 0 low voltage,electricity,,0.000000,0.0,-1.0,True
DK1 0,DK1 0 low voltage,electricity,,0.000000,0.0,-1.0,True
DK0 0 land transport EV,DK0 0 EV battery,land transport EV,,0.000000,0.0,-1.0,True
DK1 0 land transport EV,DK1 0 EV battery,land transport EV,,0.000000,0.0,-1.0,True
DK0 0 urban central heat,DK0 0 urban central heat,urban central heat,,0.000000,0.0,-1.0,True
DK1 0 urban central heat,DK1 0 urban central heat,urban central heat,,0.000000,0.0,-1.0,True
DK0 0 solid biomass for industry,DK0 0 solid biomass for industry,solid biomass for industry,,541.095890,0.0,-1.0,True
DK1 0 solid biomass for industry,DK1 0 solid biomass for industry,solid biomass for industry,,245.433790,0.0,-1.0,True
DK0 0 gas for industry,DK0 0 gas for industry,gas for industry,,155.251142,0.0,-1.0,True


In [33]:
n.links

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,underground,under_construction,tags,geometry,dc,underwater_fraction,energy to power ratio,location,reversed,length_original
Link,,,,,,,,,,,,,,,,,,,,,
relation/5487095-400-DC,DK0 0,DK1 0,,DC,0.976096,True,0,inf,600.0,0.0,...,1.0,0.0,relation/5487095,LINESTRING (10.505724427906852 55.365970143543...,1.0,0.560249,NaN,,False,171.53249
DK0 0 co2 sequestered,DK0 0 co2 stored,DK0 0 co2 sequestered,,co2 sequestered,1.000000,True,0,inf,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.00000
DK1 0 co2 sequestered,DK1 0 co2 stored,DK1 0 co2 sequestered,,co2 sequestered,1.000000,True,0,inf,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.00000
DK0 0 OCGT,DK0 0 gas,DK0 0,,OCGT,0.430000,True,0,25.0,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.00000
DK1 0 OCGT,DK1 0 gas,DK1 0,,OCGT,0.430000,True,0,25.0,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.00000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
DK1 0 methanolisation cluster,DK1 0 H2 cluster,EU methanol,,methanolisation,0.878700,True,0,20.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
DK0 0 battery charger cluster,DK0 0 cluster,DK0 0 battery cluster,,battery,1.000000,True,0,inf,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
DK0 0 battery discharger cluster,DK0 0 battery cluster,DK0 0 cluster,,battery,1.000000,True,0,inf,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,NaN


**Exporting**

In [34]:
n.export_to_netcdf(fn)


INFO:pypsa.network.io:Exported network 'Unnamed Network'saved to 'resources/DK_test/networks/base_s_2__12h_2050.nc contains: carriers, stores, buses, loads, generators, global_constraints, links


<xarray.Dataset> Size: 541kB
Dimensions:                               (snapshots: 730,
                                           investment_periods: 0,
                                           carriers_i: 118, stores_i: 32,
                                           stores_t_e_min_pu_i: 2,
                                           stores_t_e_max_pu_i: 4, buses_i: 66,
                                           loads_i: 37, loads_t_p_set_i: 10,
                                           generators_i: 65,
                                           generators_t_p_max_pu_i: 50,
                                           global_constraints_i: 3,
                                           links_i: 143,
                                           links_t_efficiency_i: 8,
                                           links_t_p_max_pu_i: 4)
Coordinates: (12/15)
  * snapshots                             (snapshots) int64 6kB 0 1 ... 728 729
  * investment_periods                    (investment_periods) object 0B 
  * carriers_i                            (carriers_i) object 944B 'AC' ... '...
  * stores_i                              (stores_i) object 256B 'co2 atmosph...
  * stores_t_e_min_pu_i                   (stores_t_e_min_pu_i) object 16B 'D...
  * stores_t_e_max_pu_i                   (stores_t_e_max_pu_i) object 32B 'D...
    ...                                    ...
  * generators_i                          (generators_i) object 520B 'DK0 0 0...
  * generators_t_p_max_pu_i               (generators_t_p_max_pu_i) object 400B ...
  * global_constraints_i                  (global_constraints_i) object 24B '...
  * links_i                               (links_i) object 1kB 'relation/5487...
  * links_t_efficiency_i                  (links_t_efficiency_i) object 64B '...
  * links_t_p_max_pu_i                    (links_t_p_max_pu_i) object 32B 'DK...
Data variables: (12/90)
    snapshots_snapshot                    (snapshots) datetime64[ns] 6kB 2013...
    snapshots_objective                   (snapshots) float64 6kB 12.0 ... 12.0
    snapshots_stores                      (snapshots) float64 6kB 12.0 ... 12.0
    snapshots_generators                  (snapshots) float64 6kB 12.0 ... 12.0
    investment_periods_objective          (investment_periods) float64 0B 
    investment_periods_years              (investment_periods) float64 0B 
    ...                                    ...
    links_energy to power ratio           (links_i) float64 1kB nan nan ... nan
    links_location                        (links_i) object 1kB '' '' ... nan nan
    links_reversed                        (links_i) bool 143B False ... True
    links_length_original                 (links_i) float64 1kB 171.5 ... nan
    links_t_efficiency                    (snapshots, links_t_efficiency_i) float64 47kB ...
    links_t_p_max_pu                      (snapshots, links_t_p_max_pu_i) float64 23kB ...
Attributes:
    network__multi_invest:  0
    network_name:           Unnamed Network
    network_pypsa_version:  0.35.2
    network_srid:           4326
    crs:                    {"_crs": "GEOGCRS[\"WGS 84\",ENSEMBLE[\"World Geo...
    meta:                   {"version": "v2025.07.0", "tutorial": false, "log...